In [ ]:
!pip install transformers datasets accelerate evaluate -U
import os
from google.colab import drive

# Montar o Drive para salvar checkpoints e métricas
drive.mount('/content/drive')

# Definir caminhos no Drive
DRIVE_PATH = "/content/drive/MyDrive/models/TinyBERT_General_4L_312D"
CHECKPOINT_DIR = os.path.join(DRIVE_PATH, "checkpoints")
METRICS_FILE = os.path.join(DRIVE_PATH, "training_metrics.csv")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
Mounted at /content/drive


In [ ]:
import json
from datasets import Dataset, DatasetDict

# Carregar o arquivo gerado anteriormente
with open("bert_qa_dataset_final_v2.json", "r", encoding="utf-8") as f:
    squad_data = json.load(f)

# Flatten do formato SQuAD para lista simples
flattened_data = []
for article in squad_data["data"]:
    for paragraph in article["paragraphs"]:
        context = paragraph["context"]
        for qa in paragraph["qas"]:
            flattened_data.append({
                "context": context,
                "question": qa["question"],
                "answers": qa["answers"],
                "id": qa["id"]
            })

# Criar Dataset do Hugging Face
raw_dataset = Dataset.from_list(flattened_data)

# Split: 90% treino, 10% teste (validação)
dataset_split = raw_dataset.train_test_split(test_size=0.1, seed=42)
tokenized_dataset = DatasetDict({
    'train': dataset_split['train'],
    'validation': dataset_split['test']
})

print(f"Dataset carregado: {len(tokenized_dataset['train'])} treio | {len(tokenized_dataset['validation'])} validação")

Dataset carregado: 13756 treio | 1529 validação


In [ ]:
from transformers import AutoTokenizer

model_name = "huawei-noah/TinyBERT_General_4L_312D"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def prepare_train_features(examples):
    # Tokenização com janelas deslizantes para contextos longos
    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=384,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")

    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized_examples.sequence_ids(i)
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        if len(answers) == 0:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            start_char = answers[0]["answer_start"]
            end_char = start_char + len(answers[0]["text"])

            token_start_index = 0
            while sequence_ids[token_start_index] != 1: token_start_index += 1
            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != 1: token_end_index -= 1

            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                tokenized_examples["start_positions"].append(cls_index)
                tokenized_examples["end_positions"].append(cls_index)
            else:
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                tokenized_examples["start_positions"].append(token_start_index - 1)
                while offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                tokenized_examples["end_positions"].append(token_end_index + 1)

    return tokenized_examples

train_dataset = tokenized_dataset["train"].map(prepare_train_features, batched=True, remove_columns=tokenized_dataset["train"].column_names)
eval_dataset = tokenized_dataset["validation"].map(prepare_train_features, batched=True, remove_columns=tokenized_dataset["validation"].column_names)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Map:   0%|          | 0/13756 [00:00<?, ? examples/s]

Map:   0%|          | 0/1529 [00:00<?, ? examples/s]

In [ ]:
import pandas as pd
from transformers import Trainer, TrainingArguments, AutoModelForQuestionAnswering, TrainerCallback

# Callback customizado para salvar métricas em CSV para o Trabalho Final
class MetricsLogCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            # Salva qualquer log que tenha 'loss' OU 'eval_loss'
            if "loss" in logs or "eval_loss" in logs:
                df = pd.DataFrame([logs])
                if not os.path.isfile(METRICS_FILE):
                    df.to_csv(METRICS_FILE, index=False)
                else:
                    df.to_csv(METRICS_FILE, mode='a', header=False, index=False)

model = AutoModelForQuestionAnswering.from_pretrained(model_name)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    eval_strategy="epoch",  # Avaliar a cada época
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,  # Ajustado para Colab Free
    gradient_accumulation_steps=2,  # Batch efetivo de 16
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    save_total_limit=2,             # Mantém apenas os 2 melhores para economizar Drive
    logging_steps=50,
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,     # Parâmetro atualizado da classe Trainer
    callbacks=[MetricsLogCallback()]
)

# Iniciar Treinamento
trainer.train()
trainer.save_model(os.path.join(DRIVE_PATH, "final_model"))

pytorch_model.bin:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/69 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight   

model.safetensors:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss
1,0.607884,0.494242
2,0.459587,0.375772
3,0.354276,0.364527


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import pandas as pd

# Reconstruir o CSV a partir do log_history do trainer (já em memória)
records = []
for entry in trainer.state.log_history:
    records.append(entry)

df_history = pd.DataFrame(records)
df_history.to_csv(METRICS_FILE, index=False)
print(f"Métricas salvas: {df_history.columns.tolist()}")
print(df_history[['epoch', 'loss', 'eval_loss']].dropna(how='all'))

Métricas salvas: ['loss', 'grad_norm', 'learning_rate', 'epoch', 'step', 'eval_loss', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second', 'train_runtime', 'train_samples_per_second', 'train_steps_per_second', 'total_flos', 'train_loss']
        epoch      loss  eval_loss
0    0.015845  4.473422        NaN
1    0.031691  2.866497        NaN
2    0.047536  2.697872        NaN
3    0.063381  2.586270        NaN
4    0.079227  2.411463        NaN
..        ...       ...        ...
188  2.962763  0.401581        NaN
189  2.978609  0.409687        NaN
190  2.994454  0.354276        NaN
191  3.000000       NaN   0.364527
192  3.000000       NaN        NaN

[193 rows x 3 columns]
